# Model Trainer — BBC News Classifier
**Fokus**: generate file `.pkl` siap pakai untuk backend FastAPI.

Notebook ini adalah versi ringkas dari notebook analisis. Tidak ada EDA, ablation study, atau visualisasi — hanya pipeline training final yang menghasilkan 3 artefak:

| File | Isi | Pemakaian |
|---|---|---|
| `model.pkl` | Stacking ensemble terlatih | Inference |
| `tfidf.pkl` | TF-IDF vectorizer | Transform input baru |
| `feature_weights.json` | Top-N kata diskriminatif per kategori (dari LinearSVC) | Word-level highlighting di frontend |

**Cara pakai**: jalankan semua sel dari atas ke bawah. Output disimpan di folder `artifacts/`.

## 1. Setup

In [ ]:
!pip install scikit-learn==1.3.2 pandas numpy nltk gdown joblib -q
import nltk
for pkg in ['stopwords', 'punkt', 'punkt_tab', 'wordnet']:
    nltk.download(pkg, quiet=True)

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score, classification_report

RANDOM_STATE = 42
os.makedirs('artifacts', exist_ok=True)

## 2. Load Dataset

Ganti path di bawah sesuai lokasi CSV-mu. Untuk Kaggle: upload BBC_news_train.csv sebagai dataset, lalu pakai path `/kaggle/input/<nama-dataset>/BBC_news_train.csv`.

In [ ]:
# Opsi A: Kaggle Dataset
# df = pd.read_csv('/kaggle/input/bbc-news/BBC_news_train.csv')

# Opsi B: Google Drive via gdown
import gdown
file_id = '1j8QnP4v5CaMVDwuo3d04_OLQzazFYigp'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'BBC_news_train.csv', quiet=False)
df = pd.read_csv('BBC_news_train.csv')

print(f'Total: {len(df)} artikel')
print(df['Category'].value_counts())

## 3. Preprocessing

In [ ]:
base_sw = set(stopwords.words('english'))
custom_sw = {
    'said', 'says', 'say', 'mr', 'mrs', 'ms', 'also', 'would', 'could',
    'one', 'two', 'three', 'four', 'five', 'year', 'years', 'people',
    'told', 'us', 'last', 'first', 'new', 'time', 'will', 'may', 'much',
    'many', 'still', 'even', 'going', 'get', 'got', 'like', 'back',
    'good', 'well', 'just', 'make', 'made', 'way', 'now', 'however',
    'though', 'although', 'despite', 'bbc', 'added', 'per', 'cent',
    'expected', 'including', 'according'
}
all_sw = base_sw | custom_sw
lemma = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|\d+|[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [lemma.lemmatize(t) for t in word_tokenize(text)
              if t not in all_sw and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['Text'].apply(preprocess)
print('Preprocessing selesai. Contoh:', df['clean_text'].iloc[0][:120], '...')

## 4. Split & TF-IDF

Konfigurasi TF-IDF dan rasio split sama persis dengan notebook analisis (yang sudah dioptimasi via ablation study).

In [ ]:
X = df['clean_text']
y = df['Category']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000,
    sublinear_tf=True,
    min_df=2
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
print(f'TF-IDF matrix: {X_train_tfidf.shape}')

## 5. Hyperparameter Tuning (Singkat)

GridSearch ringkas — hanya parameter paling berpengaruh, untuk dapat best C/alpha dengan cepat.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

# LinearSVC
grid_svc = GridSearchCV(
    LinearSVC(max_iter=3000),
    {'C': [0.1, 0.5, 1, 2]},
    cv=cv, scoring='accuracy', n_jobs=-1
)
grid_svc.fit(X_train_tfidf, y_train)
best_c_svc = grid_svc.best_params_['C']
print(f'LinearSVC best C={best_c_svc} ({grid_svc.best_score_:.4f})')

# ComplementNB
grid_cnb = GridSearchCV(
    ComplementNB(),
    {'alpha': [0.05, 0.1, 0.2, 0.5]},
    cv=cv, scoring='accuracy', n_jobs=-1
)
grid_cnb.fit(X_train_tfidf, y_train)
best_alpha = grid_cnb.best_params_['alpha']
print(f'ComplementNB best alpha={best_alpha} ({grid_cnb.best_score_:.4f})')

## 6. Train Stacking Ensemble (Final Model)

In [ ]:
base_models = [
    ('svc', LinearSVC(C=best_c_svc, max_iter=3000)),
    ('cnb', ComplementNB(alpha=best_alpha)),
    ('lr',  LogisticRegression(C=5, max_iter=2000)),
]

final_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5, n_jobs=-1
)
final_model.fit(X_train_tfidf, y_train)

# Evaluasi
y_pred = final_model.predict(X_val_tfidf)
val_acc = accuracy_score(y_val, y_pred)
print(f'Validation accuracy: {val_acc:.4f}')
print(classification_report(y_val, y_pred, digits=4))

## 7. Ekstrak Feature Weights

LinearSVC standalone dilatih lagi untuk akses koefisien per kelas. Top-N kata per kategori disimpan ke JSON yang nanti dipakai frontend untuk highlighting.

In [ ]:
interp_svc = LinearSVC(C=best_c_svc, max_iter=3000)
interp_svc.fit(X_train_tfidf, y_train)

feature_names = np.array(tfidf.get_feature_names_out())
classes = list(interp_svc.classes_)

TOP_N = 80   # ambil top-80 kata per kategori untuk highlighting
feature_weights = {}

for i, cls in enumerate(classes):
    coefs = interp_svc.coef_[i]
    # Filter: ambil hanya yang positif (relevan untuk kelas ini)
    top_idx = np.argsort(coefs)[-TOP_N:][::-1]
    feature_weights[cls] = [
        {'word': feature_names[j], 'weight': float(coefs[j])}
        for j in top_idx if coefs[j] > 0
    ]
    print(f'{cls}: top 5 = {[fw["word"] for fw in feature_weights[cls][:5]]}')

# Simpan ke JSON
with open('artifacts/feature_weights.json', 'w') as f:
    json.dump({
        'classes': classes,
        'features': feature_weights,
        'metadata': {
            'top_n': TOP_N,
            'val_accuracy': val_acc,
            'best_C_svc': best_c_svc,
            'best_alpha_cnb': best_alpha,
        }
    }, f, indent=2)
print('\nSaved: artifacts/feature_weights.json')

## 8. Simpan Model PKL

Dua file PKL:
- `tfidf.pkl` — vectorizer untuk transform input baru
- `model.pkl` — stacking classifier untuk prediksi

Compress level=3 agar size kecil tapi load cepat.

In [ ]:
joblib.dump(tfidf, 'artifacts/tfidf.pkl', compress=3)
joblib.dump(final_model, 'artifacts/model.pkl', compress=3)

# Cek ukuran
for f in ['tfidf.pkl', 'model.pkl', 'feature_weights.json']:
    size_kb = os.path.getsize(f'artifacts/{f}') / 1024
    print(f'{f:30s} {size_kb:>8.1f} KB')

## 9. Test Loading

Verifikasi PKL bisa diload kembali dan menghasilkan prediksi yang sama.

In [ ]:
# Reload
tfidf_loaded = joblib.load('artifacts/tfidf.pkl')
model_loaded = joblib.load('artifacts/model.pkl')

# Test dengan 3 kalimat
test_texts = [
    'The bank reported strong quarterly earnings and dividend increase.',
    'The Manchester United striker scored a brilliant goal in the final.',
    'A new smartphone with revolutionary AI features has launched.',
]

for txt in test_texts:
    clean = preprocess(txt)
    vec = tfidf_loaded.transform([clean])
    pred = model_loaded.predict(vec)[0]
    print(f'[{pred:14s}] {txt}')

## 10. Download Artifacts

Setelah semua sel di atas selesai, download 3 file dari folder `artifacts/`:
- `model.pkl`
- `tfidf.pkl`
- `feature_weights.json`

Pindahkan ke `backend/artifacts/` di repo GitHub-mu.

In [ ]:
# Di Kaggle: download via shutil + IPython
import shutil
shutil.make_archive('artifacts', 'zip', 'artifacts')
print('Created: artifacts.zip')
# Download artifacts.zip dari panel Output di Kaggle